# Text Preprocessing Pipeline

Applies the full preprocessing pipeline to all six Amazon review files:
- Drops constant and low-value columns
- Handles missing values (critical vs non-critical fields)
- Cleans HTML tags and entities from review text
- Combines `review_headline` + `review_body` into a single `review` column
- Derives `sentiment_label`, `helpfulness_ratio`, and date-based features
- Saves each processed file as Parquet to `data/processed/`

In [ ]:
import os
import re
import html
import pandas as pd

DATA_DIR  = "../data"
OUT_DIR   = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

FILES = {
    "Apparel":   "Apparel_review.tsv",
    "Beauty":    "Beauty_review.tsv",
    "Books":     "Books_review.tsv",
    "Furniture": "Furniture_review.tsv",
    "Mobile":    "Mobile_review.tsv",
    "Outdoors":  "Outdoors_review.tsv",
}

# Fields where a missing value means the row must be dropped
CRITICAL_FIELDS = [ "product_parent","star_rating", "review_body", "review_date"]

## Text Cleaning Function

In [2]:
def clean_text(text):
    """Remove HTML tags/entities, lowercase, normalise whitespace."""
    if not isinstance(text, str) or not text.strip():
        return None
    text = html.unescape(text)                  # decode &#34; &#39; etc.
    text = re.sub(r"<[^>]+>", " ", text)        # strip HTML tags
    text = text.lower()
    text = re.sub(r"[\t\n\r]+", " ", text)      # collapse line breaks / tabs
    text = re.sub(r" {2,}", " ", text).strip()  # collapse extra spaces
    return text if text else None

## Preprocessing Pipeline

In [5]:
def preprocess(df, source_category):
    # --- source category (from filename, more reliable than product_category column) ---
    df["source_category"] = source_category

    # --- drop constant column ---
    df = df.drop(columns=["marketplace"])

    # --- drop rows missing critical fields ---
    df = df.dropna(subset=CRITICAL_FIELDS)
    df = df[df["review_body"].str.strip().ne("")]  # drop empty review_body

    # --- fill product_title / product_category from other rows sharing product_id ---
    for col in ["product_title", "product_category"]:
        null_mask = df[col].isna()
        if null_mask.any():
            fill = df.groupby("product_id")[col].transform(
                lambda s: s.ffill().bfill()
            )
            df[col] = df[col].where(~null_mask, other=fill)
    df = df.dropna(subset=["product_title", "product_category"])

    # --- missing values: non-critical text → None, numeric → NaN ---
    df["review_headline"] = df["review_headline"].where(df["review_headline"].notna(), other=None)
    df["customer_id"]     = df["customer_id"].where(df["customer_id"].notna(), other=float("nan"))
    df["review_id"]       = df["review_id"].where(df["review_id"].notna(), other=float("nan"))

    # --- text cleaning ---
    df["review_headline"] = df["review_headline"].apply(clean_text)
    df["review_body"]     = df["review_body"].apply(clean_text)

    # drop rows where review_body became None after cleaning
    df = df.dropna(subset=["review_body"])

    # --- combine headline + body into review ---
    df["review"] = df.apply(
        lambda r: (r["review_headline"] + ". " + r["review_body"])
                  if pd.notna(r["review_headline"]) and r["review_headline"]
                  else r["review_body"],
        axis=1,
    )
    df = df.drop(columns=["review_headline", "review_body"])

    # --- sentiment label ---
    rating = df["star_rating"].astype("Int64")
    df["sentiment_label"] = pd.cut(
        rating.astype("float64"),
        bins=[0, 2, 3, 5],
        labels=["negative", "neutral", "positive"],
        right=True,
    ).astype(str)
    df = df.drop(columns=["star_rating"])

    # --- helpfulness ratio ---
    total   = df["total_votes"].astype("float64")
    helpful = df["helpful_votes"].astype("float64")
    df["helpfulness_ratio"] = (helpful / total).where(total > 0, 0).fillna(0)

    # --- date features ---
    dates = pd.to_datetime(df["review_date"], errors="coerce")
    df["review_year"]        = dates.dt.year
    df["review_month"]       = dates.dt.month
    df["review_day_of_week"] = dates.dt.day_name()

    return df

## Run Pipeline & Save

In [6]:
for category, filename in FILES.items():
    print(f"Processing {category}...")

    df = pd.read_csv(
        os.path.join(DATA_DIR, filename),
        sep="\t",
        on_bad_lines="skip",
        engine="python",
    )
    rows_before = len(df)

    df = preprocess(df, category)

    out_path = os.path.join(OUT_DIR, f"{category.lower()}_processed.parquet")
    df.to_parquet(out_path, index=False)

    print(f"  {rows_before:>9,} → {len(df):>9,} rows  ({rows_before - len(df):,} dropped)")
    print(f"  Saved to {out_path}")
    print()

Processing Apparel...
  5,877,663 → 5,876,656 rows  (1,007 dropped)
  Saved to ../data/processed/apparel_processed.parquet

Processing Beauty...
  5,090,735 → 5,090,248 rows  (487 dropped)
  Saved to ../data/processed/beauty_processed.parquet

Processing Books...
  3,101,049 → 3,101,044 rows  (5 dropped)
  Saved to ../data/processed/books_processed.parquet

Processing Furniture...
    790,987 →   790,842 rows  (145 dropped)
  Saved to ../data/processed/furniture_processed.parquet

Processing Mobile...
  5,013,957 → 5,013,710 rows  (247 dropped)
  Saved to ../data/processed/mobile_processed.parquet

Processing Outdoors...
  2,298,620 → 2,298,456 rows  (164 dropped)
  Saved to ../data/processed/outdoors_processed.parquet



## Verify Output

Quick sanity check on one processed file.

In [7]:
sample = pd.read_parquet(os.path.join(OUT_DIR, "apparel_processed.parquet"))
print(sample.shape)
print(sample.dtypes)
print()
sample[["review_id", "sentiment_label", "helpfulness_ratio",
        "review_year", "review_month", "review_day_of_week", "review"]].head(3)

(5876656, 18)
customer_id             int64
review_id                 str
product_id                str
product_parent          int64
product_title             str
product_category          str
helpful_votes           int64
total_votes             int64
vine                      str
verified_purchase         str
review_date               str
source_category           str
review                    str
sentiment_label           str
helpfulness_ratio     float64
review_year             int32
review_month            int32
review_day_of_week        str
dtype: object



,review_id,sentiment_label,helpfulness_ratio,review_year,review_month,review_day_of_week,review
0,R1KKOXHNI8MSXU,positive,0.0,2013,1,Monday,★ these really do work great with some tweakin...
1,R26SP2OPDK4HT7,positive,0.5,2014,3,Tuesday,favorite for winter. very warm!. i love this d...
2,RWQEDYAX373I1,positive,0.0,2015,7,Sunday,"great socks for the money.. nice socks, great ..."


In [8]:
# Sentiment label distribution
sample["sentiment_label"].value_counts()

sentiment_label
positive    4445410
negative     810830
neutral      620416
Name: count, dtype: int64